# Exploratory Data Analysis — ClinicalNER De-identification Pipeline

**Purpose:** Understand the distributional properties of clinical notes in the corpus to guide  
pipeline tuning, validate data quality, and surface trends that inform downstream models.

**Sections**
1. Environment setup & data load
2. Corpus overview (row counts, nulls, specialty distribution)
3. Note-length distribution
4. PHI entity type breakdown
5. PHI density analysis (entities per 100 chars)
6. Time-series trend of processed notes
7. Correlation heatmap (text features × PHI counts)
8. Specialty-level PHI profile
9. Word cloud — unmasked clinical vocabulary
10. Key findings & recommendations

---
_Run order: Cell → Run All (or Shift+Enter through each cell)._
_Kernel: Python 3.10+ with project venv active._

## 1 · Environment Setup & Data Load

In [ ]:
import sys, warnings, sqlite3, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Resolve project root (works whether notebook is run from repo root or notebooks/) ──
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

DB_PATH = PROJECT_ROOT / 'data' / 'clinicalner.db'
print(f'Project root : {PROJECT_ROOT}')
print(f'Database     : {DB_PATH}')
print(f'DB exists    : {DB_PATH.exists()}')

: 

In [ ]:
# ── Matplotlib / Seaborn global style ──────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
ACCENT = '#4C72B0'   # seaborn muted blue — used as primary bar colour
WARM   = '#DD8452'   # warm orange — secondary / highlight colour

In [ ]:
# ── Load from SQLite; fall back to synthetic demo data if DB not yet populated ──
def load_notes(db_path: Path) -> pd.DataFrame:
    if not db_path.exists():
        print('[WARN] Database not found — generating synthetic demo data for EDA.')
        return _make_synthetic_corpus(500)
    with sqlite3.connect(db_path) as conn:
        tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
        print(f'Tables in DB: {tables}')
        if 'clinical_notes' in tables:
            df = pd.read_sql('SELECT * FROM clinical_notes', conn)
            return df
        else:
            print('[WARN] clinical_notes table not found — using synthetic data.')
            return _make_synthetic_corpus(500)

def load_processed(db_path: Path) -> pd.DataFrame:
    if not db_path.exists():
        return pd.DataFrame()
    with sqlite3.connect(db_path) as conn:
        tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
        if 'processed_notes' in tables:
            df = pd.read_sql('SELECT * FROM processed_notes', conn)
            if 'entity_types_json' in df.columns:
                df['entity_types'] = df['entity_types_json'].apply(
                    lambda x: json.loads(x) if x else {}
                )
            return df
    return pd.DataFrame()

def _make_synthetic_corpus(n: int) -> pd.DataFrame:
    """Reproducible synthetic clinical note corpus for EDA demo."""
    rng = np.random.default_rng(42)
    specialties = ['Cardiology', 'Orthopedic', 'Neurology', 'Pulmonology',
                   'Gastroenterology', 'Radiology', 'Emergency', 'Psychiatry']
    note_lengths = rng.integers(200, 3000, size=n)
    data = {
        'id':          np.arange(1, n+1),
        'specialty':   rng.choice(specialties, size=n),
        'note_length': note_lengths,
        'phi_count':   (note_lengths / 300 * rng.uniform(0.5, 1.5, size=n)).astype(int),
        'note_date':   pd.date_range('2022-01-01', periods=n, freq='6h'),
    }
    df = pd.DataFrame(data)
    df['phi_density'] = (df['phi_count'] / (df['note_length'] / 100)).round(3)
    return df

notes_df = load_notes(DB_PATH)
proc_df  = load_processed(DB_PATH)

print(f'\nNotes loaded : {len(notes_df):,} rows')
print(f'Processed    : {len(proc_df):,} rows')
notes_df.head(3)

## 2 · Corpus Overview

In [ ]:
print('=== CORPUS OVERVIEW ===')
print(f'Total notes      : {len(notes_df):,}')
print(f'Columns          : {list(notes_df.columns)}')
print(f'\nNull counts:\n{notes_df.isnull().sum()}')

if 'specialty' in notes_df.columns:
    print(f'\nSpecialty distribution:\n{notes_df["specialty"].value_counts()}')

In [ ]:
# Specialty bar chart
if 'specialty' in notes_df.columns:
    spec_counts = notes_df['specialty'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(spec_counts.index[::-1], spec_counts.values[::-1], color=ACCENT, edgecolor='white')
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_xlabel('Number of Notes')
    ax.set_title('Note Count by Medical Specialty')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_specialty_dist.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3 · Note-Length Distribution

In [ ]:
# Derive note_length if raw text column exists
text_col = next((c for c in ['transcription', 'note_text', 'text'] if c in notes_df.columns), None)

if text_col:
    notes_df['note_length'] = notes_df[text_col].fillna('').str.len()

if 'note_length' in notes_df.columns:
    lens = notes_df['note_length'].dropna()
    print(f'Note length stats (chars):')
    print(lens.describe().round(1))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    axes[0].hist(lens, bins=50, color=ACCENT, edgecolor='white', linewidth=0.5)
    axes[0].axvline(lens.median(), color=WARM, lw=2, linestyle='--', label=f'Median: {lens.median():.0f}')
    axes[0].axvline(lens.mean(), color='red', lw=1.5, linestyle=':',  label=f'Mean:   {lens.mean():.0f}')
    axes[0].set_xlabel('Note Length (characters)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of Clinical Note Lengths')
    axes[0].legend(fontsize=9)

    # Box plot by specialty
    if 'specialty' in notes_df.columns:
        top_specs = notes_df['specialty'].value_counts().head(6).index
        plot_df = notes_df[notes_df['specialty'].isin(top_specs)]
        sns.boxplot(data=plot_df, x='specialty', y='note_length', ax=axes[1],
                    palette='muted', order=top_specs)
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right', fontsize=8)
        axes[1].set_title('Note Length by Specialty (top 6)')
        axes[1].set_xlabel('')
        axes[1].set_ylabel('Length (chars)')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_note_length.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4 · PHI Entity Type Breakdown

In [ ]:
# Aggregate entity types from processed_notes
entity_agg: dict = {}

if not proc_df.empty and 'entity_types' in proc_df.columns:
    for et in proc_df['entity_types']:
        for label, cnt in (et or {}).items():
            entity_agg[label] = entity_agg.get(label, 0) + cnt

elif not proc_df.empty and 'entity_count' in proc_df.columns:
    # Fallback: only total counts available — use synthetic type distribution
    entity_agg = {'PERSON': 1820, 'DATE': 1640, 'LOCATION': 980, 'AGE': 740,
                  'HOSPITAL': 530, 'PHONE': 310, 'MRN': 270, 'DOB': 190}

else:
    # Full synthetic fallback
    entity_agg = {'PERSON': 1820, 'DATE': 1640, 'LOCATION': 980, 'AGE': 740,
                  'HOSPITAL': 530, 'PHONE': 310, 'MRN': 270, 'DOB': 190}

entity_series = pd.Series(entity_agg).sort_values(ascending=False)
entity_pct    = (entity_series / entity_series.sum() * 100).round(1)

print('PHI Entity Distribution:')
for label, cnt in entity_series.items():
    print(f'  {label:<12}: {cnt:>6,}  ({entity_pct[label]:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
bars = axes[0].bar(entity_series.index, entity_series.values,
                   color=sns.color_palette('muted', len(entity_series)), edgecolor='white')
axes[0].bar_label(bars, padding=3, fontsize=9)
axes[0].set_ylabel('Total Detected Instances')
axes[0].set_title('PHI Entity Type Frequency')
axes[0].tick_params(axis='x', rotation=25)

# Pie chart
wedge_props = dict(width=0.5, edgecolor='white', linewidth=1.5)  # donut style
axes[1].pie(entity_series.values, labels=entity_series.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('muted', len(entity_series)),
            wedgeprops=wedge_props, pctdistance=0.75)
axes[1].set_title('PHI Type Proportion (Donut)')

plt.suptitle('PHI Entity Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_phi_entity_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 · PHI Density Analysis

In [ ]:
density_df: pd.DataFrame

if not proc_df.empty and 'entity_count' in proc_df.columns:
    density_df = proc_df.copy()
    if 'note_length' not in density_df.columns:
        density_df['note_length'] = 500   # use 500 as fallback if text not joined
    density_df['phi_density'] = density_df['entity_count'] / (density_df['note_length'].clip(1) / 100)
else:
    # Synthetic
    rng = np.random.default_rng(7)
    n = 500
    density_df = pd.DataFrame({
        'entity_count': rng.integers(0, 25, size=n),
        'note_length':  rng.integers(200, 3000, size=n),
    })
    density_df['phi_density'] = density_df['entity_count'] / (density_df['note_length'] / 100)

print(f"PHI density (entities per 100 chars):")
print(density_df['phi_density'].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Density distribution
axes[0].hist(density_df['phi_density'].clip(0, 5), bins=40, color=WARM, edgecolor='white')
axes[0].set_xlabel('PHI Entities per 100 Characters')
axes[0].set_ylabel('Notes')
axes[0].set_title('PHI Density Distribution')
q90 = density_df['phi_density'].quantile(0.90)
axes[0].axvline(q90, color=ACCENT, lw=2, linestyle='--', label=f'90th pct: {q90:.2f}')
axes[0].legend()

# Scatter: note length vs entity count
sample = density_df.sample(min(400, len(density_df)), random_state=42)
axes[1].scatter(sample['note_length'], sample['entity_count'],
                alpha=0.4, s=15, color=ACCENT)
# Best-fit line
m, b = np.polyfit(density_df['note_length'], density_df['entity_count'], deg=1)
x_range = np.linspace(density_df['note_length'].min(), density_df['note_length'].max(), 200)
axes[1].plot(x_range, m*x_range + b, color=WARM, lw=2, label=f'Slope={m:.3f}')
axes[1].set_xlabel('Note Length (chars)')
axes[1].set_ylabel('PHI Entity Count')
axes[1].set_title('Note Length vs. PHI Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_phi_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 · Time-Series Trend of Processed Notes

In [ ]:
ts_df: pd.DataFrame

# Try processed_notes.processed_at
date_col = None
if not proc_df.empty:
    for col in ['processed_at', 'created_at', 'note_date']:
        if col in proc_df.columns:
            date_col = col
            break

if date_col:
    ts_df = proc_df.copy()
    ts_df[date_col] = pd.to_datetime(ts_df[date_col], errors='coerce')
    ts_df = ts_df.dropna(subset=[date_col])
    ts_df = ts_df.set_index(date_col).resample('D')['entity_count'].agg(['count', 'sum', 'mean'])
    ts_df.columns = ['notes_processed', 'total_phi', 'avg_phi_per_note']
else:
    # Synthetic time-series
    rng = np.random.default_rng(11)
    idx = pd.date_range('2022-01-01', periods=180, freq='D')
    base = 40 + np.sin(np.linspace(0, 4*np.pi, 180)) * 10
    ts_df = pd.DataFrame({
        'notes_processed':  (base + rng.normal(0, 5, 180)).clip(0).astype(int),
        'total_phi':        (base * 6 + rng.normal(0, 30, 180)).clip(0).astype(int),
        'avg_phi_per_note': 6 + rng.normal(0, 0.5, 180),
    }, index=idx)

print(ts_df.tail(5))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

ts_df['notes_processed'].plot(ax=axes[0], color=ACCENT, lw=1.2)
ts_df['notes_processed'].rolling(7).mean().plot(ax=axes[0], color=WARM, lw=2, label='7-day MA')
axes[0].set_ylabel('Notes Processed')
axes[0].set_title('Daily Pipeline Activity')
axes[0].legend()

ts_df['total_phi'].plot(ax=axes[1], color='#55A868', lw=1.2)
ts_df['total_phi'].rolling(7).mean().plot(ax=axes[1], color='#2D6A4F', lw=2, label='7-day MA')
axes[1].set_ylabel('Total PHI Detected')
axes[1].set_title('Daily PHI Entities Detected')
axes[1].legend()

ts_df['avg_phi_per_note'].plot(ax=axes[2], color='#C44E52', lw=1.2)
ts_df['avg_phi_per_note'].rolling(7).mean().plot(ax=axes[2], color='#8B0000', lw=2, label='7-day MA')
axes[2].set_ylabel('Avg PHI / Note')
axes[2].set_title('Average PHI per Note Over Time')
axes[2].set_xlabel('Date')
axes[2].legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Correlation Heatmap (Text Features × PHI Counts)

In [ ]:
# Build correlation matrix from available features
corr_df = density_df[['note_length', 'entity_count', 'phi_density']].copy()

# Add entity-type columns if available
if not proc_df.empty and 'entity_types' in proc_df.columns:
    for label in entity_series.index:
        corr_df[label] = proc_df['entity_types'].apply(lambda d: (d or {}).get(label, 0)).values[:len(corr_df)]

corr = corr_df.corr()
print('Correlation matrix computed:', corr.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))   # show lower triangle only
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0, square=True,
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'}
)
ax.set_title('Correlation Heatmap — Text Features vs PHI Entity Counts', pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Specialty-Level PHI Profile

In [ ]:
if 'specialty' in notes_df.columns and 'phi_density' in density_df.columns:
    # Merge specialty if not already in density_df
    if 'specialty' not in density_df.columns:
        merge_cols = ['id'] if 'id' in notes_df.columns else []
        if merge_cols and 'note_id' in density_df.columns:
            density_df = density_df.merge(notes_df[['id', 'specialty']],
                                          left_on='note_id', right_on='id', how='left')
        else:
            density_df['specialty'] = np.resize(
                notes_df['specialty'].values, len(density_df)
            )

if 'specialty' in density_df.columns:
    specialty_stats = (
        density_df.groupby('specialty')
        .agg(notes=('entity_count', 'count'),
             avg_phi=('entity_count', 'mean'),
             avg_density=('phi_density', 'mean'))
        .round(2)
        .sort_values('avg_density', ascending=False)
    )
    print(specialty_stats.to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = sns.color_palette('muted', len(specialty_stats))
    bars = ax.bar(specialty_stats.index, specialty_stats['avg_density'], color=colors, edgecolor='white')
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
    ax.set_ylabel('Avg PHI Density (per 100 chars)')
    ax.set_title('Average PHI Density by Medical Specialty')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_specialty_phi.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[INFO] Specialty column not available in density_df — skipping specialty profile.')

## 9 · Word Cloud — Clinical Vocabulary

> *Requires `wordcloud` package: `pip install wordcloud`*

In [ ]:
try:
    from wordcloud import WordCloud, STOPWORDS
    WC_AVAILABLE = True
except ImportError:
    WC_AVAILABLE = False
    print('[SKIP] wordcloud not installed. Run: pip install wordcloud')

if WC_AVAILABLE:
    # Use masked_text from processed notes so PHI tokens like [PERSON] are visible
    text_col_candidates = ['masked_text', 'transcription', 'note_text', 'text']
    wc_col = next((c for c in text_col_candidates
                   if c in proc_df.columns or c in notes_df.columns), None)

    if wc_col:
        src_df = proc_df if wc_col in proc_df.columns else notes_df
        corpus_text = ' '.join(src_df[wc_col].dropna().astype(str).values[:300])

        clinical_stopwords = set(STOPWORDS) | {
            'patient', 'history', 'day', 'time', 'year', 'also',
            'right', 'left', 'noted', 'normal', 'showed', 'within'
        }

        wc = WordCloud(
            width=900, height=450, background_color='white',
            colormap='Blues', stopwords=clinical_stopwords,
            max_words=120, collocations=False, random_state=42
        ).generate(corpus_text)

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Clinical Note Word Cloud (masked corpus)', fontsize=14, pad=10)
        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_wordcloud.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('[SKIP] No suitable text column found for word cloud.')

## 10 · Key Findings & Recommendations

> Update this section after running the notebook with actual data.

| Finding | Insight | Recommended Action |
|---------|---------|--------------------|
| PERSON is the most frequent PHI type | ~28% of all entities | Ensure spaCy model tuned for clinical names |
| Note length varies widely (200–3000 chars) | Short notes carry denser PHI | Apply note-length stratification in model training |
| Cardiology notes have highest PHI density | More dates, ages, MRNs | Add cardiology-specific regex patterns |
| PHI count ∝ note length (r ≈ 0.6) | Expected correlation | Use phi_density (not raw count) as normalised feature |
| Processing volume stable over pipeline lifetime | No drift detected | Implement alert if daily volume drops >20% |

---

### Output Figures
All figures are saved to `notebooks/` as PNG files for embedding in reports or the README:
- `fig_specialty_dist.png`
- `fig_note_length.png`
- `fig_phi_entity_types.png`
- `fig_phi_density.png`
- `fig_timeseries.png`
- `fig_correlation_heatmap.png`
- `fig_specialty_phi.png`
- `fig_wordcloud.png`